# Perf-sweep explorer

This notebook explores the perf sweep at `SWEEP_DIR` (set in the next cell). Run all cells for the default views; the interactive sections (filter view and GPU saturation panel) near the end lets you filter the matrix without writing matplotlib code.

When the sweep carries the SOR-1025 CPU thread-count axis (`cpu_n1` … `cpu_n128`), the default-plot cell also renders `omp_scaling.png`, `omp_efficiency.png`, and `gpu_cpu_crossover_heatmap.png`.

Works in both JupyterLab and classic Jupyter. Requires the packages listed in `tests/perf/analyze/requirements.txt` (`pandas`, `numpy`, `matplotlib`, `jupyter`, `ipywidgets`).

CPU reference defaults to `cpu_n64` (the knee of the OpenMP curve on this hardware — see SOR-1025's `omp_scaling.png`). `cpu_n128` is available in the dropdown but its SMT contention makes it a misleading default.

In [ ]:
# Configuration — edit SWEEP_DIR to point at any past sweep.
# PRIOR_SWEEP_DIR is optional; set to None to skip pre/post comparison cells.
import os
import sys
from pathlib import Path

# Add repo root to sys.path so `tests.perf.analyze` imports work when the
# notebook is launched from anywhere.
_HERE = Path.cwd()
for _candidate in [_HERE] + list(_HERE.parents):
    if (_candidate / 'tests' / 'perf' / 'analyze').is_dir():
        REPO_ROOT = _candidate
        break
else:
    raise RuntimeError('could not locate repo root from ' + str(_HERE))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SWEEP_DIR = os.environ.get('SWEEP_DIR') or str(REPO_ROOT / 'tests/perf/perf-scaling-sweep-20260519-post-sor1019')
PRIOR_SWEEP_DIR = os.environ.get('PRIOR_SWEEP') or str(REPO_ROOT / 'tests/perf/perf-scaling-sweep-20260519')

print('SWEEP_DIR        =', SWEEP_DIR)
print('PRIOR_SWEEP_DIR  =', PRIOR_SWEEP_DIR)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tests.perf.analyze import load, plot, compare

df = load.load_sweep(SWEEP_DIR)
df_prior = load.load_sweep(PRIOR_SWEEP_DIR) if PRIOR_SWEEP_DIR else None
label = load.sweep_label(SWEEP_DIR)
subtitle = plot._figure_subtitle(load.capture_env(SWEEP_DIR))
print(f'loaded {len(df)} cells from {label}')
df.head(8)

## Default plot set

Same five plots `tests/perf/analyze/plot.py` writes to `<sweep>/plots/`, rendered inline below.

In [ ]:
# Re-render the default plot set into a temp dir so we can show them inline,
# AND so opening the .png path printed below in another tab works.
import tempfile
TMP_PLOTS = Path(tempfile.mkdtemp(prefix='perf_explore_'))
plot.plot_wall_vs_length(df, label, subtitle, TMP_PLOTS / 'wall_vs_length.png')
plot.plot_per_step_vs_length(df, label, subtitle, TMP_PLOTS / 'per_step_vs_length.png')
plot.plot_gpu_vs_cpu_speedup(df, label, subtitle, TMP_PLOTS / 'gpu_vs_cpu_speedup.png')
plot.plot_component_breakdown(df, label, subtitle, TMP_PLOTS / 'component_breakdown.png')
plot.plot_gpu_utilization(df, label, subtitle, TMP_PLOTS / 'gpu_utilization.png')

# SOR-1037: GPU saturation vs rank-count panel — rendered when the sweep
# carries at least two gpu_n<k> rank counts.
extra = []
if len(plot._gpu_rank_counts(df)) >= 2:
    plot.plot_gpu_saturation(df, TMP_PLOTS, label=label, subtitle=subtitle)
    extra.append('gpu_saturation_vs_rank.png')

# SOR-1025: thread-count axis plots — rendered only when the sweep
# carries cpu_n<k> for k != 128.
if len(plot._cpu_thread_counts(df)) >= 2:
    plot.plot_omp_scaling(df, label, subtitle, TMP_PLOTS / 'omp_scaling.png')
    plot.plot_omp_efficiency(df, label, subtitle, TMP_PLOTS / 'omp_efficiency.png')
    extra.append('omp_scaling.png')
    extra.append('omp_efficiency.png')
    if plot._gpu_rank_counts(df):
        plot.plot_gpu_cpu_crossover_heatmap(df, label, subtitle, TMP_PLOTS / 'gpu_cpu_crossover_heatmap.png')
        extra.append('gpu_cpu_crossover_heatmap.png')

from IPython.display import Image, display
for name in ['wall_vs_length.png', 'per_step_vs_length.png', 'gpu_vs_cpu_speedup.png',
             'component_breakdown.png', 'gpu_utilization.png'] + extra:
    display(Image(filename=str(TMP_PLOTS / name)))


## Interactive filtering

Pick `case`, `config`, `grid`, one or more `length`s, and (for cpu_n\<k\> sweeps captured per SOR-1025) one or more thread counts; the plot below regenerates on every change. Skipped cells (`skipped_estimated_wall` set, no measurement) are excluded from this view — see the static `omp_scaling.png` for projected-position markers.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

case_w = widgets.Dropdown(
    options=sorted(df['case'].dropna().unique()),
    value=sorted(df['case'].dropna().unique())[0],
    description='case',
)
config_w = widgets.SelectMultiple(
    options=sorted(df['config'].dropna().unique()),
    value=tuple(sorted(df['config'].dropna().unique())),
    description='config',
    rows=min(6, len(df['config'].dropna().unique())),
)
# SOR-1025: thread-count multi-select. Acts as a CPU-side filter
# stacked on top of config_w — only cpu_n<k> with k in selection are
# kept; gpu_* configs pass through unfiltered.
_cpu_ks = sorted({int(c[5:]) for c in df['config'].dropna().unique() if c.startswith('cpu_n')})
threads_w = widgets.SelectMultiple(
    options=_cpu_ks if _cpu_ks else [128],
    value=tuple(_cpu_ks) if _cpu_ks else (128,),
    description='cpu_n',
    rows=min(8, max(1, len(_cpu_ks))),
)
# SOR-1041: CPU reference dropdown. Default = cpu_n64 (OpenMP knee on
# the production host; see plot.CPU_REFERENCE and SOR-1025). Lists every
# cpu_n<k> present in the sweep so operators can switch back to
# cpu_n128 to inspect SMT contention without editing code.
_cpu_configs = sorted(
    [c for c in df['config'].dropna().unique() if c.startswith('cpu_n')],
    key=lambda c: int(c[5:]),
) or [plot.CPU_REFERENCE]
cpu_ref_w = widgets.Dropdown(
    options=_cpu_configs,
    value=plot.CPU_REFERENCE if plot.CPU_REFERENCE in _cpu_configs else _cpu_configs[-1],
    description='CPU ref',
)
grid_w = widgets.Dropdown(
    options=sorted(df['grid'].dropna().unique()),
    value='1x',
    description='grid',
)
length_w = widgets.SelectMultiple(
    options=['1h', '6h', '24h', '4d'],
    value=('1h', '6h', '24h', '4d'),
    description='length',
)
metric_w = widgets.Dropdown(
    options=['wall', 'per_step', 'U_L', 'total_simulation_time'],
    value='wall',
    description='metric',
)

out = widgets.Output()

LENGTH_SECONDS = {'1h': 3600, '6h': 21600, '24h': 86400, '4d': 345600}

def _filter_configs(configs_selected, threads_selected):
    """Cross-filter config_w + threads_w. cpu_n<k> kept iff both
    selected; gpu_* configs kept iff selected in config_w."""
    keep = []
    threads_set = set(threads_selected or [])
    for cfg in configs_selected:
        if cfg.startswith('cpu_n'):
            try:
                k = int(cfg[5:])
            except ValueError:
                continue
            if k in threads_set:
                keep.append(cfg)
        else:
            keep.append(cfg)
    return keep

def _redraw(_=None):
    out.clear_output(wait=True)
    with out:
        case = case_w.value
        configs = _filter_configs(list(config_w.value), list(threads_w.value)) \
            or sorted(df['config'].dropna().unique())
        # SOR-1041: always include the chosen CPU reference so the
        # baseline line is drawn even if the operator narrowed
        # threads_w / config_w. Other selections still pass through.
        if cpu_ref_w.value not in configs:
            configs = list(configs) + [cpu_ref_w.value]
        grid = grid_w.value
        lengths = list(length_w.value) or ['1h', '6h', '24h', '4d']
        metric = metric_w.value

        sub = df[(df['case'] == case) & (df['grid'] == grid) & (df['config'].isin(configs)) & (df['length'].isin(lengths))]
        # Skipped rows have wall=NaN; the interactive plot drops them.
        sub = sub[sub[metric].notna()] if metric in sub.columns else sub
        if sub.empty:
            print('no cells match the current filter.')
            return
        fig, ax = plt.subplots(figsize=(9, 5))
        for cfg in configs:
            s = sub[sub['config'] == cfg].copy()
            if s.empty:
                continue
            s['_x'] = s['length'].map(LENGTH_SECONDS)
            s = s.sort_values('_x')
            ys = s[metric].values
            if metric == 'per_step':
                ys = ys * 1000.0
            ax.plot(s['_x'].values, ys, marker='o', label=cfg,
                    color=plot.CONFIG_COLORS.get(cfg, '#333'))
        ax.set_xscale('log')
        ax.set_xticks([LENGTH_SECONDS[L] for L in ['1h', '6h', '24h', '4d']])
        ax.set_xticklabels(['1h', '6h', '24h', '4d'])
        ax.set_xlabel('simulation length')
        ylabel = metric + (' (ms)' if metric == 'per_step' else ' (s)')
        ax.set_ylabel(ylabel)
        ax.set_title(f'{case} | grid={grid}')
        ax.legend(fontsize=8, ncol=2)
        ax.grid(True, alpha=0.3)
        plt.show()

for w in (case_w, config_w, threads_w, cpu_ref_w, grid_w, length_w, metric_w):
    w.observe(_redraw, names='value')

ui = widgets.VBox([
    widgets.HBox([case_w, config_w, threads_w]),
    widgets.HBox([grid_w, length_w, cpu_ref_w]),
    metric_w,
])
display(ui)
display(out)
_redraw()

## GPU saturation vs rank count (SOR-1037)

Per-case wall-time bars at the longest measured length, colored green when adding ranks helps (`gpu_n<k+1> wall <= gpu_n<k> wall`) and red when it hurts (`> `). The single-GPU SM% (p50) overlay on the right Y axis surfaces cases where the GPU is underloaded — useful for spotting workloads whose per-step compute can't amortize the inter-rank halo-exchange overhead.

Use the checkbox below to toggle the SM% overlay on/off. The `SM% threshold` slider tunes the `UNDERLOADED` annotation threshold (default 30%).


In [ ]:
# SOR-1037: interactive GPU saturation panel.
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

sat_overlay_w = widgets.Checkbox(
    value=True,
    description='show GPU saturation overlay',
    indent=False,
)
sat_threshold_w = widgets.FloatSlider(
    value=30.0, min=0.0, max=100.0, step=1.0,
    description='SM% threshold',
    continuous_update=False,
)
sat_out = widgets.Output()

def _redraw_saturation(_=None):
    sat_out.clear_output(wait=True)
    with sat_out:
        entries = plot.compute_gpu_saturation_panel(
            df, sm_threshold_pct=sat_threshold_w.value,
        )
        if not entries:
            print('Sweep has no (case, grid) combo with ≥ 2 gpu_n<k> measurements.')
            return
        nrows, ncols, figsize = plot._layout(len(entries))
        fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
        flat = axes.flatten()
        for i, e in enumerate(entries):
            ax = flat[i]
            x = np.arange(len(e['rank_counts']))
            ax.bar(x, e['walls'], width=0.55, color=e['bar_colors'],
                   edgecolor='black', linewidth=0.5)
            ax.set_xticks(x)
            ax.set_xticklabels([f'gpu_n{g}' for g in e['rank_counts']])
            ax.set_xlabel('rank count')
            ax.set_ylabel('wall time (s)')
            ymax = max(e['walls']) if e['walls'] else 1.0
            ax.set_ylim(0, ymax * 1.25)
            ax.grid(True, axis='y', alpha=0.3)
            grid_suffix = f' [{e["grid"]}]' if e['grid'] != '1x' else ''
            ax.set_title(
                f'{e["case"]}{grid_suffix}  |  best: {e["best_config"]}  '
                f'({e["length"]})',
                fontsize=9,
            )
            if sat_overlay_w.value:
                ax2 = ax.twinx()
                sm_x = x + 0.28
                sm_plot = [s if s == s else 0.0 for s in e['sms']]
                ax2.bar(sm_x, sm_plot, width=0.15, color='#666666',
                        alpha=0.75, edgecolor='black', linewidth=0.3)
                ax2.set_ylim(0, 100)
                ax2.set_ylabel('GPU SM% (p50)')
                if e['underloaded']:
                    n1_sm = e['sms'][0]
                    ax.text(
                        0.5, 0.94,
                        'UNDERLOADED — n2 adds overhead without compute payoff\n'
                        f'(SM%={n1_sm:.0f} < threshold={sat_threshold_w.value:.0f})',
                        transform=ax.transAxes, ha='center', va='top',
                        fontsize=7, color=plot.SATURATION_BAR_RED,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                  edgecolor=plot.SATURATION_BAR_RED, alpha=0.9),
                    )
        for j in range(len(entries), nrows * ncols):
            flat[j].axis('off')
        fig.tight_layout()
        plt.show()

for w in (sat_overlay_w, sat_threshold_w):
    w.observe(_redraw_saturation, names='value')

display(widgets.HBox([sat_overlay_w, sat_threshold_w]))
display(sat_out)
_redraw_saturation()


## Pre/post side-by-side

Two sweep dirs → per-case absolute + percent deltas as side-by-side bars.

In [ ]:
if df_prior is None or df_prior.empty:
    print('PRIOR_SWEEP_DIR not set or empty — skipping comparison view.')
else:
    cmp_grid = widgets.Dropdown(options=['1x', '4x'], value='1x', description='grid')
    _cmp_cpu_configs = sorted(
        [c for c in df['config'].dropna().unique() if c.startswith('cpu_n')],
        key=lambda c: int(c[5:]),
    )
    cmp_config = widgets.Dropdown(
        options=['gpu_n2', 'gpu_n1'] + (_cmp_cpu_configs or ['cpu_n128']),
        value='gpu_n2',
        description='config',
    )
    cmp_length = widgets.Dropdown(options=['1h', '6h', '24h', '4d'], value='4d', description='length')
    cmp_metric = widgets.Dropdown(
        options=['wall', 'total_simulation_time', 'U_L', 'per_step'],
        value='wall', description='metric',
    )
    cmp_out = widgets.Output()

    def _redraw_cmp(_=None):
        cmp_out.clear_output(wait=True)
        with cmp_out:
            grid = cmp_grid.value
            config = cmp_config.value
            length = cmp_length.value
            metric = cmp_metric.value

            cases = [c for c in plot.DEFAULT_CASE_ORDER
                     if c in set(df_prior['case'].unique()) | set(df['case'].unique())]
            scale = 1000.0 if metric == 'per_step' else 1.0
            ylabel = metric + (' (ms)' if metric == 'per_step' else ' (s)')
            xs = np.arange(len(cases))
            width = 0.4
            pre_vals = []
            post_vals = []
            for c in cases:
                pre_row = df_prior[(df_prior['case']==c) & (df_prior['grid']==grid) & (df_prior['length']==length) & (df_prior['config']==config)]
                post_row = df[(df['case']==c) & (df['grid']==grid) & (df['length']==length) & (df['config']==config)]
                pre_vals.append(float(pre_row.iloc[0][metric]) * scale if not pre_row.empty else np.nan)
                post_vals.append(float(post_row.iloc[0][metric]) * scale if not post_row.empty else np.nan)
            pre_vals = np.array(pre_vals)
            post_vals = np.array(post_vals)
            fig, (ax_abs, ax_pct) = plt.subplots(1, 2, figsize=(14, 5))
            ax_abs.bar(xs - width/2, pre_vals, width, label=f'prior ({load.sweep_label(PRIOR_SWEEP_DIR)})', color='#999')
            ax_abs.bar(xs + width/2, post_vals, width, label=f'current ({load.sweep_label(SWEEP_DIR)})', color='#4C72B0')
            ax_abs.set_xticks(xs)
            ax_abs.set_xticklabels([c.replace('case_prod_', '') for c in cases], rotation=20, ha='right')
            ax_abs.set_ylabel(ylabel)
            ax_abs.set_title(f'absolute — {metric} ({grid} {config} {length})')
            ax_abs.legend(fontsize=8)
            ax_abs.grid(True, axis='y', alpha=0.3)

            with np.errstate(divide='ignore', invalid='ignore'):
                pct = np.where(pre_vals != 0, (post_vals - pre_vals) / pre_vals * 100.0, np.nan)
            colors = ['#55A467' if (v == v and v < 0) else '#C44E52' for v in pct]
            ax_pct.bar(xs, pct, color=colors)
            ax_pct.axhline(0, color='black', linewidth=0.7)
            ax_pct.set_xticks(xs)
            ax_pct.set_xticklabels([c.replace('case_prod_', '') for c in cases], rotation=20, ha='right')
            ax_pct.set_ylabel('Δ %')
            ax_pct.set_title(f'percent — {metric} ({grid} {config} {length})')
            ax_pct.grid(True, axis='y', alpha=0.3)
            plt.tight_layout()
            plt.show()

    for w in (cmp_grid, cmp_config, cmp_length, cmp_metric):
        w.observe(_redraw_cmp, names='value')
    display(widgets.HBox([cmp_grid, cmp_config, cmp_length, cmp_metric]))
    display(cmp_out)
    _redraw_cmp()

## Mechanical delta table (`compare.py`)

Same shape as the SUMMARY.md `Delta vs prior baseline` table — rendered inline.

In [ ]:
from IPython.display import Markdown
if PRIOR_SWEEP_DIR:
    md = compare.render_delta_table(
        prior_sweep=PRIOR_SWEEP_DIR,
        current_sweep=SWEEP_DIR,
        grid='1x', length='4d', config='gpu_n2',
    )
    display(Markdown(md))
else:
    print('PRIOR_SWEEP_DIR not set — skipping delta table.')